# Phase 2 — Outage Event Selection (EAGLE-I)

**Goal:** Load EAGLE-I records, identify sharp-onset non-weather outage events,
and define treatment windows for traffic feature extraction.

**Key design constraints:**
- Only non-weather-concurrent outages (equipment failures, grid faults, rolling blackouts)
- Sharp onset: >=10% customers_out change within <=30 min
- Treatment windows: 2h pre-outage (baseline), during, 2h post-restoration

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from src.data.load_eagle_i import load_eagle_i_local

In [ ]:
# ── Load EAGLE-I data ─────────────────────────────────────────────
# Point this at wherever you saved EAGLE-I downloads
DATA_PATH = "../data/raw/eagle_i/harris_tx_2024.parquet"

eagle = load_eagle_i_local(DATA_PATH)
print(f"Records: {len(eagle):,}")
print(f"Date range: {eagle['timestamp'].min()} to {eagle['timestamp'].max()}")
eagle.head()

In [ ]:
# ── Find sharp-onset events ────────────────────────────────────────
eagle['pct_out'] = (eagle['customers_out'] / eagle['customers_total']) * 100
eagle['delta_pct'] = eagle['pct_out'].diff()

onsets = eagle[eagle['delta_pct'] >= 10].copy()
print(f"Sharp-onset events (>=10% jump in 15min): {len(onsets)}")
onsets[['timestamp', 'fips', 'customers_out', 'pct_out', 'delta_pct']].head(10)

In [ ]:
# ── Define treatment windows ───────────────────────────────────────
from datetime import timedelta

windows = []
for _, event in onsets.iterrows():
    t0 = event['timestamp']
    windows.append({
        'event_id': f"{event['fips']}_{t0.strftime('%Y%m%d_%H%M')}",
        'fips': event['fips'],
        'window_start': t0 - timedelta(hours=2),
        'outage_start': t0,
        'window_end': t0 + timedelta(hours=4),  # 2h outage + 2h post
    })

windows_df = pd.DataFrame(windows)
print(f"Treatment windows: {len(windows_df)}")
windows_df.head()

In [ ]:
# ── Save event index ──────────────────────────────────────────────
windows_df.to_parquet("../data/processed/eagle_i_events.parquet", index=False)
print("✅ Saved event index")

## Next Steps
Once NPMRDS data arrives, load these treatment windows in
notebook `03_npmrds_feature_engineering.ipynb` and extract
traffic features for each window.